# 01_supervised_baselines

Baseline supervised models (Logistic Regression + Random Forest)

In [ ]:

# ===============================
# 01_supervised_baselines.ipynb
# ===============================

# -----------------------------
# 1️⃣ Imports
# -----------------------------
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, auc, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set(style="whitegrid")

# -----------------------------
# 2️⃣ Load preprocessed data
# -----------------------------
# Ensure we are in repo root (adjust if needed)
print("cwd:", os.getcwd())

assert os.path.exists("data/processed/X_train.npy"), "Missing X_train.npy in data/processed"
X_train = np.load("data/processed/X_train.npy", allow_pickle=True)
X_test  = np.load("data/processed/X_test.npy", allow_pickle=True)
y_train = np.load("data/processed/y_train.npy", allow_pickle=True)
y_test  = np.load("data/processed/y_test.npy", allow_pickle=True)

# Load feature pipeline (from preprocess step)
pipeline_path = "models/preprocess.pkl"
if not os.path.exists(pipeline_path):
    raise FileNotFoundError(f"Pipeline not found at {pipeline_path}. Expected preprocess pipeline saved earlier.")
pipeline = joblib.load(pipeline_path)

# -----------------------------
# 3️⃣ Logistic Regression
# -----------------------------
lr = LogisticRegression(max_iter=1000, class_weight='balanced')
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
y_proba_lr = lr.predict_proba(X_test)[:,1]

print("📌 Logistic Regression Classification Report")
print(classification_report(y_test, y_pred_lr))

roc_auc_lr = roc_auc_score(y_test, y_proba_lr)
precision_lr, recall_lr, _ = precision_recall_curve(y_test, y_proba_lr)
pr_auc_lr = auc(recall_lr, precision_lr)
print(f"ROC-AUC: {roc_auc_lr:.4f}, PR-AUC: {pr_auc_lr:.4f}")

# Confusion matrix
cm_lr = confusion_matrix(y_test, y_pred_lr)
try:
    tn, fp, fn, tp = cm_lr.ravel()
except:
    tn = fp = fn = tp = 0
fpr_per_1000_lr = (fp / len(y_test)) * 1000
print(f"False Positives per 1000 hosts: {fpr_per_1000_lr:.2f}")

# Save model
os.makedirs("models", exist_ok=True)
joblib.dump(lr, "models/logistic_regression.pkl")

# -----------------------------
# 4️⃣ Random Forest
# -----------------------------
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight='balanced')
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:,1]

print("\n📌 Random Forest Classification Report")
print(classification_report(y_test, y_pred_rf))

roc_auc_rf = roc_auc_score(y_test, y_proba_rf)
precision_rf, recall_rf, _ = precision_recall_curve(y_test, y_proba_rf)
pr_auc_rf = auc(recall_rf, precision_rf)
print(f"ROC-AUC: {roc_auc_rf:.4f}, PR-AUC: {pr_auc_rf:.4f}")

cm_rf = confusion_matrix(y_test, y_pred_rf)
try:
    tn, fp, fn, tp = cm_rf.ravel()
except:
    tn = fp = fn = tp = 0
fpr_per_1000_rf = (fp / len(y_test)) * 1000
print(f"False Positives per 1000 hosts: {fpr_per_1000_rf:.2f}")

# Save model
joblib.dump(rf, "models/random_forest.pkl")

# -----------------------------
# 5️⃣ Feature Importance (Random Forest)
# -----------------------------
# Attempt to reconstruct feature names:
# We assume the preprocessing pipeline is Pipeline([("preprocess", ColumnTransformer(...))])
ct = pipeline.named_steps["preprocess"]

# Get categorical transformer (OneHotEncoder) and numeric feature names
cat_key = None
for name, trans, cols in ct.transformers_:
    if name == "cat" or (hasattr(trans, "get_feature_names_out") and isinstance(trans, (type(ct.named_transformers_.get("cat", None))))):
        cat_key = name
        cat_cols = cols
        break

# Fallback: try to read categorical columns from ct._feature_names_in (if present)
try:
    categorical_features = ct.transformers_[0][2]
except Exception:
    categorical_features = ["Protocol","Flag","Family","netflow_bucket"]

numeric_features = ["Time","Clusters","BTC","USD","Netflow_Bytes","bytes_per_second",
                    "port_risk","btc_flag","usd_flag","threat_score","Port"]

# Extract OHE feature names safely
ohe = None
if "cat" in ct.named_transformers_:
    ohe = ct.named_transformers_["cat"]
elif len([t for t in ct.named_transformers_.values() if hasattr(t, "get_feature_names_out")]) > 0:
    # pick first transformer with method
    for v in ct.named_transformers_.values():
        if hasattr(v, "get_feature_names_out"):
            ohe = v
            break

if ohe is not None:
    try:
        ohe_features = ohe.get_feature_names_out(categorical_features)
    except Exception:
        # fallback to generic names
        ohe_features = [f"cat_{i}" for i in range(len(ohe.get_feature_names_out()))]
else:
    ohe_features = [f"cat_{i}" for i in range(len(categorical_features))]

all_features = np.concatenate([ohe_features, numeric_features])

import pandas as pd
feature_importances = rf.feature_importances_
# if X_train is sparse, convert to array for shape check
try:
    n_cols = X_train.shape[1]
except Exception:
    try:
        n_cols = X_train.toarray().shape[1]
    except Exception:
        n_cols = len(all_features)

# Align lengths
if len(feature_importances) != len(all_features):
    # Try to trim/pad
    min_len = min(len(feature_importances), len(all_features))
    feature_importances = feature_importances[:min_len]
    all_features = all_features[:min_len]

feature_importance_df = pd.DataFrame({"feature": all_features, "importance": feature_importances})
feature_importance_df = feature_importance_df.sort_values("importance", ascending=False)

# Plot top 20
plt.figure(figsize=(10,6))
sns.barplot(x="importance", y="feature", data=feature_importance_df.head(20))
plt.title("Top 20 Feature Importances (Random Forest)")
plt.tight_layout()
plt.show()

# -----------------------------
# 6️⃣ Correlation Matrix Check
# -----------------------------
# Numeric features only: attempt to extract numeric slice from X_train
# If sparse, convert to array (may be large)
X_train_arr = None
try:
    # works if dense numpy
    X_train_arr = X_train
    if hasattr(X_train_arr, "toarray"):
        X_train_arr = X_train_arr.toarray()
except Exception:
    try:
        X_train_arr = X_train.toarray()
    except Exception:
        raise RuntimeError("Cannot convert X_train to array for correlation matrix.")

# numeric slice: assume numeric features are last len(numeric_features)
num_len = len(numeric_features)
if X_train_arr.shape[1] >= num_len:
    numeric_df = pd.DataFrame(X_train_arr[:, -num_len:], columns=numeric_features)
    corr_matrix = numeric_df.corr()

    plt.figure(figsize=(12,10))
    sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm")
    plt.title("Correlation Matrix — Numeric Features")
    plt.show()

    # Identify highly correlated pairs
    high_corr = np.where(corr_matrix.abs() > 0.9)
    high_corr_pairs = [(numeric_df.columns[x], numeric_df.columns[y])
                       for x,y in zip(*high_corr) if x!=y and x<y]
    print("Highly correlated numeric pairs (r>0.9):", high_corr_pairs)
else:
    print("Numeric slice not available in X_train - shape too small.")
